# 01 — Data Pipeline: Building the TfL Bike × Tube Strike Panel

This notebook builds the modelling-ready dataset used in our causal analysis of the effect of London Underground strikes on Santander Cycle hire usage.

**What we do here:**
1. Aggregate raw TfL journey CSVs into a station-hour panel
2. Load bike station coordinates from the TfL BikePoint API
3. Fetch tube station locations and served Underground lines
4. Spatially map each bike station to nearby tube stations and the lines they serve
5. Load the FOI strike calendar and expand to daily indicators
6. Attach binary strike exposure to the station-hour panel
7. Join weather, calendar, and spatial covariates
8. Aggregate to H3 hexagonal cell-day level (the modelling unit)
9. Filter to the overlap region and save the final basefile

**Data sources:**
- TfL cycle hire journey data: https://cycling.data.tfl.gov.uk/#!usage-stats%2F (free download, weekly CSVs)
- TfL BikePoint API: https://api.tfl.gov.uk/BikePoint
- TfL StopPoint API: https://api.tfl.gov.uk/StopPoint/Mode/tube
- Strike calendar FOI: https://tfl.gov.uk/corporate/transparency/freedom-of-information/foi-request-detail?referenceId=FOI-2596-1819
- Open-Meteo historical weather archive: https://archive-api.open-meteo.com


## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import h3
from scipy.spatial import cKDTree

from tfl_utils_updated import (
    aggregate_folder_to_parquet,
    combine_parquet_parts,
    load_bike_station_locations,
    load_tube_stations_foi,
    build_station_line_map,
    expand_strikes_daily,
    attach_strikes_to_base,
)
from covariate_join_utils_v2 import build_bike_panel_with_covariates_from_panel

## Paths

Edit these to match your local project directory.

In [2]:
# ── Project paths — edit for your machine ─────────────────────────────────────
PROJECT    = Path(r"E:/tfl_project")

JOURNEY_FOLDER  = PROJECT / "data"            # raw TfL journey CSVs
BIKEPOINTS_JSON = PROJECT / "data/BikePoint.json"
STRIKES_CSV     = PROJECT / "strikes/strikes.csv"
AGG_PARTS_DIR   = PROJECT / "agg_parts_parquet"   # intermediate per-file parquets
OUT_DIR         = PROJECT / "outputs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
AGG_PARTS_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Aggregate Journey CSVs to Station-Hour Panel

TfL publish one CSV per week of journeys. Each row is a single bike hire.
We read every CSV in the journey folder, aggregate to **station × hour** trip counts,
and write one parquet per CSV to `AGG_PARTS_DIR` to keep peak memory bounded.

**Modelling decision:** We count `trips_start` (journeys departing from each station)
rather than arrivals. Departures measure demand from a specific location — exactly
what we want to understand when asking whether strikes push people to take bikes.

In [4]:
# ── 1a: Aggregate each file into a per-file parquet ──────────────────────────
# This processes one file at a time — memory cost is bounded to a single weekly CSV
written = aggregate_folder_to_parquet(
    folder_path = JOURNEY_FOLDER,
    out_dir     = AGG_PARTS_DIR,
    freq        = "h",          # hourly aggregation
    side        = "start",      # count departures
    max_station_id = 5000,      # discard legacy logical terminal IDs (>5000)
)
print(f"Written {len(written)} parquet files")

Found 145 files to process
  Processing 01aJourneyDataExtract10Jan16-23Jan16.csv ... ✓
  Processing 01bJourneyDataExtract24Jan16-06Feb16.csv ... ✓
  Processing 02aJourneyDataExtract07Fe16-20Feb2016.csv ... ✓
  Processing 02bJourneyDataExtract21Feb16-05Mar2016.csv ... ✓
  Processing 03JourneyDataExtract06Mar2016-31Mar2016.csv ... ✓
  Processing 04JourneyDataExtract01Apr2016-30Apr2016.csv ... ✓
  Processing 05JourneyDataExtract01May2016-17May2016.csv ... ✓
  Processing 06JourneyDataExtract18May2016-24May2016.csv ... ✓
  Processing 07JourneyDataExtract25May2016-31May2016.csv ... ✓
  Processing 08JourneyDataExtract01Jun2016-07Jun2016.csv ... ✓
  Processing 09JourneyDataExtract08Jun2016-14Jun2016.csv ... ✓
  Processing 100JourneyDataExtract07Mar2018-13Mar2018.csv ... ✓
  Processing 101JourneyDataExtract14Mar2018-20Mar2018.csv ... ✓
  Processing 102JourneyDataExtract21Mar2018-27Mar2018.csv ... ✓
  Processing 103JourneyDataExtract28Mar2018-03Apr2018.csv ... ✓
  Processing 104JourneyDataExtrac

In [ ]:
# ── 1b: Merge all per-file parquets into a single station-hour panel ──────────
base = combine_parquet_parts(AGG_PARTS_DIR)

print(f"Panel shape: {base.shape}")
print(f"Unique stations: {base['station_id'].nunique():,}")
print(f"Time range: {base['ts'].min()} → {base['ts'].max()}")
base.head()

## Step 2 — Load Bike Station Coordinates

We load lat/lon from a BikePoint JSON dump. The TfL BikePoint API returns all
docking stations with their current coordinates. We use this to:
- Filter out station IDs with no matching location (legacy or retired stations)
- Compute spatial proximity to tube stations in Step 4

In [ ]:
bike_stations = load_bike_station_locations(BIKEPOINTS_JSON)
print(f"Bike stations loaded: {len(bike_stations)}")
bike_stations.head()

Bike stations loaded: 800


,station_id,lat,lon,station_name
0,1,51.529163,-0.109970,"River Street , Clerkenwell"
1,2,51.499606,-0.197574,"Phillimore Gardens, Kensington"
2,3,51.521283,-0.084605,"Christopher Street, Liverpool Street"
3,4,51.530059,-0.120973,"St. Chad's Street, King's Cross"
4,5,51.493130,-0.156876,"Sedding Street, Sloane Square"


In [ ]:
# Join coordinates onto the panel — filter out ~5% of rows with no location match
base = base.merge(
    bike_stations[["station_id", "lat", "lon", "station_name"]],
    on="station_id", how="inner",
)
print(f"Panel after filtering to known stations: {base.shape}")

Panel after filtering to known stations: (9206267, 6)


## Step 3 — Fetch Tube Stations and Lines

We pull all Underground stop points from the TfL StopPoint API.
Only true Underground lines are retained (not DLR, Overground, or Elizabeth line),
since the FOI strike data covers Underground industrial action only.

In [ ]:
tube_stations_foi = load_tube_stations_foi("Stations_20180921.csv")

print(f"Tube stations: {len(tube_stations_foi)}")

Loaded 270 Underground stations, 382 station-line pairs
Tube stations: 382


## Step 4 — Map Bike Stations to Tube Lines

We define a bike station as "connected" to an Underground line if it is within
**800 metres** of any tube station served by that line.

**Why 800m?** This is a standard walking catchment radius consistent with TfL's
own accessibility modelling and roughly 10 minutes on foot. It is conservative
enough to ensure the exposed stations genuinely serve the displaced commuter
population while not being so broad as to dilute the treatment.

Any bike station with no tube station within 800m falls back to the single
nearest tube stop — this affects very few central London stations.

In [ ]:
station_line_map = build_station_line_map(
    bike_stations = bike_stations,
    tube_stations_foi = tube_stations_foi,
    radius_m      = 800,
)

# Sanity check: how many lines does each bike station serve?
lines_per_station = station_line_map.groupby("station_id")["affected_line"].nunique()
print("Lines per bike station:")
print(lines_per_station.describe().round(2))
station_line_map.head()

Station-line map: 800 bike stations × 11 lines = 2247 pairs
Lines per bike station:
count    800.00
mean       2.81
std        1.90
min        1.00
25%        1.00
50%        2.00
75%        4.00
max        8.00
Name: affected_line, dtype: float64


,station_id,tube_station_name,affected_line,dist_m
0,1,Angel,northern_line,515.337440
1,2,High Street Kensington,circle_line,483.629210
2,2,High Street Kensington,district_line,483.629210
3,3,Liverpool Street,central_line,535.726330
4,3,Moorgate,circle_line,437.384066


## Step 5 — Load Strike Calendar and Expand to Daily

The strike data comes from a Freedom of Information request to TfL (FOI-2596-1819),
covering Underground strike action 2014–2018. The raw table records start and end
dates per affected line. We expand this into one row per strike day per line.

In [ ]:
strike_data = pd.read_csv(STRIKES_CSV)
print(f"Raw strike events: {len(strike_data)}")
strike_data.head()

Raw strike events: 22


,date_start,date_end,affected_line
0,07/11/18,07/11/18,central_line
1,05/10/18,05/10/18,central_line
2,26/09/18,29/09/18,piccadilly_line
3,22/08/18,24/08/18,central_line
4,12/07/18,13/07/18,central_line


In [ ]:
strikes_daily = expand_strikes_daily(strike_data)
print(f"Strike-day-line records: {len(strikes_daily)}")
print(f"Unique strike dates: {strikes_daily['date'].nunique()}")
print(f"Lines affected:", sorted(strikes_daily["affected_line"].unique()))

Strike-day-line records: 44
Unique strike dates: 20
Lines affected: ['bakerloo_line', 'central_line', 'circle_line', 'district_line', 'hammersmith-city_line', 'hammersmith_city_line', 'jubilee_line', 'metropolitan_line', 'northern_line', 'piccadilly_line', 'victoria_line', 'waterloo-city_line']


e:\tfl_project\tfl_utils_updated.py:454: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s["date_start"] = pd.to_datetime(s["date_start"], dayfirst=True, errors="coerce")
e:\tfl_project\tfl_utils_updated.py:455: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s["date_end"]   = pd.to_datetime(s["date_end"],   dayfirst=True, errors="coerce")


## Step 6 — Attach Strike Exposure to the Station-Hour Panel

A station-hour is labelled `strike_exposed = 1` if any Underground line
that the station is connected to (within 800m) is on strike that day.

**Treatment rate at station-hour level** is very low (~0.5%). This extreme
imbalance will matter for the causal estimators we choose later.

In [ ]:
base_with_strikes = attach_strikes_to_base(
    base             = base,
    strikes_daily    = strikes_daily,
    station_line_map = station_line_map,
)

treatment_rate = base_with_strikes["strike_exposed"].mean()
n_treated      = base_with_strikes["strike_exposed"].sum()
print(f"Treatment rate (station-hour): {treatment_rate:.4%}")
print(f"Treated station-hours: {n_treated:,} of {len(base_with_strikes):,}")

# Save the station-hour basefile with strike exposure
base_with_strikes.to_parquet(OUT_DIR / "base_station_hour.parquet", index=False)
print("Saved → base_station_hour.parquet")

Treatment rate (station-hour): 0.7152%
Treated station-hours: 65,846 of 9,206,267
Saved → base_station_hour.parquet


In [ ]:
base_with_strikes.columns
base_with_strikes.head()

,station_id,ts,trips_start,lat,lon,station_name,strike_exposed
0,1,2016-01-10 09:00:00,4,51.529163,-0.10997,"River Street , Clerkenwell",0
1,1,2016-01-10 10:00:00,1,51.529163,-0.10997,"River Street , Clerkenwell",0
2,1,2016-01-10 11:00:00,2,51.529163,-0.10997,"River Street , Clerkenwell",0
3,1,2016-01-10 12:00:00,2,51.529163,-0.10997,"River Street , Clerkenwell",0
4,1,2016-01-10 13:00:00,2,51.529163,-0.10997,"River Street , Clerkenwell",0


## Step 7 — Join Covariates

We join three groups of covariates onto the station-hour panel using
`covariate_join_utils.py`:

- **Weather**: temperature, humidity, precipitation, wind speed, cloud cover
  (hourly, from Open-Meteo archive API, cached as parquet)
- **Calendar**: hour, day-of-week, month, year, is_weekend, is_am_peak,
  is_pm_peak, is_bank_holiday, is_school_holiday
- **Spatial**: distance to nearest tube station, tube stations within 500m/1km,
  cycle infrastructure density from OpenStreetMap

These are the **observed confounders** in our causal model — variables that
affect both treatment probability and the outcome.

In [ ]:
# This step fetches weather from the Open-Meteo API and caches results locally.
# On first run it will take ~20 minutes depending on the number of grid cells.
# Subsequent runs use the cache and complete in seconds.

build_bike_panel_with_covariates_from_panel(
    bike_path  = str(OUT_DIR / "base_station_hour.parquet"),
    out_path   = str(OUT_DIR / "bike_hourly_with_covariates.parquet"),
    start_date = "2016-01-01",
    end_date   = "2019-12-31",
    grid_km    = 1.0,
    include_daylight = True,
  #  osm_json_path    = None,  # set to "osm_cycle_lanes.json" if available
)
print("Saved → bike_hourly_with_covariates.parquet")

InvalidOperationError: conversion from `datetime[μs]` to `i32` failed in column 'ts' for 104858 out of 104858 values: [2017-12-21 21:00:00, 2017-12-21 23:00:00, … 2017-12-22 18:00:00]

Did not show all failed cases as there were too many.

## Step 8 — Aggregate to H3 Cell-Day Level

Station-hour is too granular for our causal estimators — with ~0.5% treatment
rate across 9M rows, the effective information for treatment effect estimation
is extremely limited.

We aggregate to **H3 hexagonal cells at resolution 8** (~460m edge length).
This serves two purposes:
1. Reduces the dataset to ~142,000 cell-day rows — tractable for panel regression
2. Defines a more natural geographic unit for commuter substitution: a cell
   captures a local neighbourhood rather than a single bike dock

The treatment variable becomes `frac_exposed` — the fraction of stations in
the cell that are strike-exposed — capturing dose variation within the treatment.

In [ ]:
bf = pd.read_parquet(OUT_DIR / "bike_hourly_with_covariates.parquet")

# Assign each bike station to an H3 cell using vectorized operation
bf["h3_cell"] = [h3.latlng_to_cell(lat, lon, 8) for lat, lon in zip(bf["lat"], bf["lon"])]

# Aggregate to cell-day
bf["day"] = pd.to_datetime(bf["trips_start"]).dt.date

cell_day = (
    bf.groupby(["h3_cell", "day"])
    .agg(
        total_trips           = ("ts", "sum"),
        frac_exposed          = ("strike_exposed", "mean"),
        n_stations            = ("station_id", "nunique"),
        temperature_2m        = ("temperature_2m", "mean"),
        precipitation         = ("precipitation", "mean"),
        cloud_cover           = ("cloud_cover", "mean"),
        wind_speed_10m        = ("wind_speed_10m", "mean"),
        is_weekend            = ("is_weekend", "first"),
        is_am_peak            = ("is_am_peak", "first"),
        is_pm_peak            = ("is_pm_peak", "first"),
        is_bank_holiday       = ("is_bank_holiday", "first"),
        is_school_holiday     = ("is_school_holiday", "first"),
        strike_severity_daily_frac = ("strike_severity_daily_frac", "first"),
        days_to_next_strike   = ("days_to_next_strike", "first"),
        days_since_last_strike= ("days_since_last_strike", "first"),
        month                 = ("month", "first"),
        year                  = ("year", "first"),
        doy                   = ("doy", "first"),
        lat                   = ("lat", "mean"),
        lon                   = ("lon", "mean"),
    )
    .reset_index()
)

cell_day["day"]              = pd.to_datetime(cell_day["day"])
cell_day["y_per_station_log1p"] = np.log1p(cell_day["total_trips"] / cell_day["n_stations"])
cell_day["treated"]          = (cell_day["frac_exposed"] > 0).astype(int)

print(f"Cell-day panel: {cell_day.shape}")
print(f"Unique cells: {cell_day['h3_cell'].nunique()}")
print(f"Treatment rate: {cell_day['treated'].mean():.4%}")
print(f"Treated cell-days: {cell_day['treated'].sum():,}")

Cell-day panel: (185625, 24)
Unique cells: 174
Treatment rate: 0.7100%
Treated cell-days: 1,318


## Step 9 — Recompute Cell-Level Tube Proximity Features

Station-level proximity features (`dist_nearest_tube_km`, `n_tube_within_500m`)
should be recomputed from H3 cell centroids rather than averaged across stations.

**Why?** Averaging station-level values conflates the cell's tube access with
the number of stations it contains, reintroducing the `n_stations` confounding
problem we solved for the outcome. Computing from the centroid treats the cell
as a geographic unit in its own right.

In [ ]:
def load_tube_stations(csv_path: str) -> pd.DataFrame:
    """
    Load all 271 London Underground station coordinates from the TfL FOI CSV.
    Filters out Overground, DLR, TfL Rail and Tramlink entries.
    """
    df = pd.read_csv(csv_path)
    lu = df[df["NETWORK"] == "London Underground"].copy()
    return pd.DataFrame({
        "name": lu["NAME"].str.strip(),
        "lat":  lu["y"].astype(float),   # column y = latitude
        "lon":  lu["x"].astype(float),   # column x = longitude
        "lines": lu["LINES"].str.strip(),
    }).reset_index(drop=True)

# Replace the hardcoded block with this single line
_TUBE_STATIONS = load_tube_stations("Stations_20180921.csv")

print(f"Loaded {len(_TUBE_STATIONS)} Underground stations")
# → Loaded 271 Underground stations

def compute_cell_tube_features(cell_ids, tube_stations, radii_km=(0.5, 1.0)):
    """Compute tube proximity features from H3 cell centroids."""
    centroids = pd.DataFrame([
        {"h3_cell": cid, "lat": h3.cell_to_latlng(cid)[0], "lon": h3.cell_to_latlng(cid)[1]}
        for cid in cell_ids
    ])
    tube_coords  = np.radians(tube_stations[["lat", "lon"]].values)
    tree         = cKDTree(tube_coords)
    cell_coords  = np.radians(centroids[["lat", "lon"]].values)

    nearest_rad, _ = tree.query(cell_coords, k=1)
    centroids["dist_nearest_tube_km"] = nearest_rad * 6371.0

    for r in radii_km:
        counts = tree.query_ball_point(cell_coords, r=r / 6371.0, return_length=True)
        centroids[f"n_tube_within_{int(r*1000)}m"] = counts

    return centroids.drop(columns=["lat", "lon"])

tube_feats = compute_cell_tube_features(
    cell_ids      = cell_day["h3_cell"].unique().tolist(),
    tube_stations = _TUBE_STATIONS,
)

# Drop any old station-averaged proximity cols and replace with centroid-based ones
cell_day = cell_day.drop(
    columns=[c for c in ["dist_nearest_tube_km","n_tube_within_500m","n_tube_within_1km"]
             if c in cell_day.columns]
).merge(tube_feats, on="h3_cell", how="left")

print(cell_day[["dist_nearest_tube_km", "n_tube_within_500m"]].describe().round(3))

Loaded 271 Underground stations
       dist_nearest_tube_km  n_tube_within_500m
count            185625.000          185625.000
mean                  0.795               0.414
std                   0.558               0.628
min                   0.049               0.000
25%                   0.405               0.000
50%                   0.610               0.000
75%                   1.032               1.000
max                   2.700               4.000


## Step 10 — Filter to Overlap Region

The **positivity (overlap) assumption** for causal inference requires that
every unit has a non-zero probability of both treatment and control.

Bike stations more than 2km from any tube stop have P(T=1|X) ≈ 0 by
construction — they are never near a striking line. Including them provides
no causal information and introduces noise. We exclude them.

**Result:** 97% of cells that remain are ever-treated at least once across
the panel. This confirms the data has a clean panel structure where the same
cells move in and out of treatment, which is ideal for DiD estimation.

In [ ]:
# Filter to cells within 2km of a tube station
cell_day = cell_day[cell_day["n_tube_within_500m"] >= 1].copy()

ever_treated = cell_day.groupby("h3_cell")["treated"].max()
print(f"Cells ever treated  : {(ever_treated == 1).sum()}")
print(f"Cells never treated : {(ever_treated == 0).sum()}")
print(f"Fraction treated    : {(ever_treated == 1).mean():.3f}")
print(f"\nFinal panel: {cell_day.shape}")

# Add date string for regression fixed effects
cell_day["date_str"] = cell_day["day"].astype(str)

# Save the final modelling basefile
cell_day.to_parquet(OUT_DIR / "cell_day_final.parquet", index=False)
print("\nSaved → cell_day_final.parquet")

Cells ever treated  : 127
Cells never treated : 2
Fraction treated    : 0.984

Final panel: (137629, 28)

Saved → cell_day_final.parquet


: 